In [9]:


import numpy
import matplotlib.pyplot as plt
import pandas as pd

import AILibs

def load_region_data(file_path, mw_column_name):
    # 1. Load the CSV
    df = pd.read_csv(file_path)
    
    # 2. Convert Datetime from string to datetime objects
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    
    # 3. Set index and sort chronologically (crucial for time series!)
    df = df.set_index('Datetime').sort_index()
    
    # 4. Rename the column to a standardized name
    df = df.rename(columns={mw_column_name: 'demand'})

    x = numpy.array(df["demand"].values)
    
    return x

def create_sliding_windows(series, window_size, horizon=1):
    X, y = [], []
    # Convert dataframe column to numpy array
    
    for i in range(len(series) - window_size - horizon + 1):
        # Input: sequence from i to i + window_size
        X.append(series[i : i + window_size])
        # Target: the value(s) immediately following the window
        y.append(series[i + window_size + horizon - 1])

    X = numpy.array(X)
    X = numpy.expand_dims(X, axis=-1)  # add feature dimension
    y = numpy.array(y)

    return X, y


# load datset
dataset_root_path = "/users/michal/datasets/hourly-energy-consumption/PJME_hourly.csv"

x_orig = load_region_data(dataset_root_path, 'PJME_MW')


In [10]:

# split train / test
train_ratio = 0.8
window_size = 24 * 7         # one week of hourly data
prediction_horizon = 24      # predict the next day


x_train = x_orig[:int(len(x_orig)*train_ratio)]
x_test  = x_orig[int(len(x_orig)*train_ratio):]


# normalization
x_mean = x_train.mean()
x_std = x_train.std()

x_train_norm = (x_train - x_mean) / (x_std + 1e-8)
x_test_norm  = (x_test - x_mean) / (x_std + 1e-8)

# create windows
x_train_window, y_train = create_sliding_windows(x_train_norm, window_size, prediction_horizon)
x_test_window, y_test   = create_sliding_windows(x_test_norm, window_size, prediction_horizon)

print("x_train_window = ", x_train_window.shape )

x_train_window =  (116101, 168, 1)


In [11]:
# obtain features for train and test sets

features_extractor = AILibs.features.Catch22Features(x_train_window[0:10])

z_batch_size    = 512

n_batches = 100

x_train_window = x_train_window[0:z_batch_size*n_batches]  # limit to 10k samples for faster testing
y_train = y_train[0:z_batch_size*n_batches]
y_train = numpy.expand_dims(y_train, axis=-1)  # add feature dimension

x_test_window = x_test_window[0:z_batch_size*n_batches]  # limit to 10k samples for faster testing
y_test = y_test[0:z_batch_size*n_batches]
y_test = numpy.expand_dims(y_test, axis=-1)  # add feature dimension

# training features

z_train = []

for n in range(n_batches):
    start_idx = n * z_batch_size

    z = features_extractor.forward(x_train_window[start_idx : start_idx + z_batch_size])
    z_train.append(z)

    print(n, "/", n_batches)

z_train = numpy.concatenate(z_train, axis=0)





z_test = []

n_batches = 20
for n in range(n_batches):
    start_idx = n * z_batch_size

    z = features_extractor.forward(x_test_window[start_idx : start_idx + z_batch_size])
    z_test.append(z)

    print(n, "/", n_batches)

z_test = numpy.concatenate(z_test, axis=0)
y_test = y_test[0:z_batch_size*n_batches] 





0 / 100
1 / 100
2 / 100
3 / 100
4 / 100
5 / 100
6 / 100
7 / 100
8 / 100
9 / 100
10 / 100
11 / 100
12 / 100
13 / 100
14 / 100
15 / 100
16 / 100
17 / 100
18 / 100
19 / 100
20 / 100
21 / 100
22 / 100
23 / 100
24 / 100
25 / 100
26 / 100
27 / 100
28 / 100
29 / 100
30 / 100
31 / 100
32 / 100
33 / 100
34 / 100
35 / 100
36 / 100
37 / 100
38 / 100
39 / 100
40 / 100
41 / 100
42 / 100
43 / 100
44 / 100
45 / 100
46 / 100
47 / 100
48 / 100
49 / 100
50 / 100
51 / 100
52 / 100
53 / 100
54 / 100
55 / 100
56 / 100
57 / 100
58 / 100
59 / 100
60 / 100
61 / 100
62 / 100
63 / 100
64 / 100
65 / 100
66 / 100
67 / 100
68 / 100
69 / 100
70 / 100
71 / 100
72 / 100
73 / 100
74 / 100
75 / 100
76 / 100
77 / 100
78 / 100
79 / 100
80 / 100
81 / 100
82 / 100
83 / 100
84 / 100
85 / 100
86 / 100
87 / 100
88 / 100
89 / 100
90 / 100
91 / 100
92 / 100
93 / 100
94 / 100
95 / 100
96 / 100
97 / 100
98 / 100
99 / 100
0 / 20
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20


In [12]:
forest = AILibs.forest.RandomForest()    

print("x_train.shape = ", z_train.shape)
print("y_train.shape = ", y_train.shape)    

forest.fit(z_train, y_train, max_depth=8, num_trees=256, num_subsamples=8192, num_random_candidates=16)


x_train.shape =  (51200, 22)
y_train.shape =  (51200, 1)


In [13]:
print("Predicting with Random Forest...")
y_pred = forest.predict_batch(z_test)


metrics = AILibs.metrics.regression_evaluation(y_test, y_pred)


for key, value in metrics.items():
    print(f"{key}: {value}")

Predicting with Random Forest...
n_samples: 10240
n_features: 1
mse: 0.69971
rmse: 0.83649
mae: 0.65528
medae: 0.53849
max_ae: 2.95061
mape: 246.71208
r2: 0.26686
residual_mean: -0.1666
residual_std: 0.81977
mse_1sigma: 0.17654
mse_2sigma: 0.51726
mse_3sigma: 0.68158
